# TeClass: Sarvam LLM with chain-of-thought prompt

Zero-shot classification on TeClass using sarvam-105b-conversations with chain-of-thought prompting.

Model: sarvam-105b-conversations
Prompt: ask the model to reason step by step before outputting a label (JSON format).
Samples: 400 stratified test examples.
Expected cost: ~₹4

## 1. Setup

In [2]:
!pip install -q requests tqdm

In [3]:
import os
import re
import json
import time
import random
from pathlib import Path

import pandas as pd
import numpy as np
import requests
from tqdm.auto import tqdm

from sklearn.metrics import f1_score, confusion_matrix

try:
    from google.colab import userdata
    SARVAM_KEY = userdata.get('SARVAM_API_KEY')
except Exception:
    SARVAM_KEY = os.environ.get('SARVAM_API_KEY', 'PASTE_SARVAM_KEY_HERE')

SARVAM_URL = "https://api.sarvam.ai/v1/chat/completions"
SARVAM_MODEL = "sarvam-105b-conversations"

r = requests.post(
    SARVAM_URL,
    headers={
        "api-subscription-key": SARVAM_KEY,
        "Content-Type": "application/json",
    },
    json={
        "model": SARVAM_MODEL,
        "messages": [{"role": "user", "content": "Say ok."}],
        "temperature": 0,
        "max_tokens": 50,
    },
    timeout=30,
)
print("status:", r.status_code)
if r.status_code == 200:
    print("sarvam:", r.json()["choices"][0]["message"]["content"].strip())
else:
    print("error:", r.text[:300])

status: 200
sarvam: Ok.


## 2. Load test set

In [4]:
def load(path):
    df = pd.read_csv(path)
    cols = {c.lower(): c for c in df.columns}
    art  = next(cols[c] for c in ["article", "text", "body", "news_article"] if c in cols)
    head = next(cols[c] for c in ["headline", "title", "news_headline"] if c in cols)
    lab  = next(cols[c] for c in ["label", "category", "class"] if c in cols)
    df = df[[art, head, lab]].rename(columns={art: "article", head: "headline", lab: "label"})
    df["label"] = df["label"].astype(str).str.upper().str.strip()
    df = df[df["label"].isin(["HREL", "MREL", "LREL"])].reset_index(drop=True)
    return df

test_df = load("test.csv")

LABELS = ["HREL", "MREL", "LREL"]
lab2id = {"HREL": 0, "MREL": 1, "LREL": 2}

print("test:", test_df.shape)
print(test_df["label"].value_counts())

test: (3927, 3)
label
MREL    1523
HREL    1278
LREL    1126
Name: count, dtype: int64


## 3. Chain-of-thought prompt

Asks the model to reason in 3 steps before answering, then output JSON.

In [5]:
DEFS = """Categories:
- HREL (Highly Related): headline faithfully summarises the article's main event.
- MREL (Moderately Related): headline is loosely related, covers a side event or weak inference.
- LREL (Least Related): headline is misleading, sensational, or clickbait."""

def cut(text, n=2200):
    text = str(text)
    return text if len(text) <= n else text[:n] + " [...]"

def p_cot(a, h):
    return (f"Classify the relevance between this Telugu news article and headline.\n\n"
            f"{DEFS}\n\nArticle: {cut(a)}\n\nHeadline: {h}\n\n"
            f"Think step by step:\n"
            f"1. In one sentence, what is the article about?\n"
            f"2. In one sentence, what does the headline claim?\n"
            f"3. Compare and decide the label.\n\n"
            f'Reply as JSON only: {{"reason": "...", "label": "HREL|MREL|LREL"}}')

print(p_cot(test_df.iloc[0]["article"], test_df.iloc[0]["headline"])[:600])

Classify the relevance between this Telugu news article and headline.

Categories:
- HREL (Highly Related): headline faithfully summarises the article's main event.
- MREL (Moderately Related): headline is loosely related, covers a side event or weak inference.
- LREL (Least Related): headline is misleading, sensational, or clickbait.

Article: అనంతపురం జిల్లాలో అమానుష ఘటన చోటు చేసుకుంది . క్లాస్ రూమ్ లో అల్లరి చేస్తున్నారనే కారణంతో చిన్నపిల్లలని కూడా చూడకుండా ఒక పాఠశాల ప్రధానోపాధ్యాయురాలు వారిని తాళ్లతో కట్టి బంధించి హింసించింది . వివరాల్లోకి వెళితే అనంతపురం జిల్లా కదిరి మున్సిపాలిటీ పరిధిలోన


## 4. Sarvam wrapper + parser

In [6]:
LABEL_RE = re.compile(r"\b(HREL|MREL|LREL)\b", re.IGNORECASE)

def parse_label(text):
    if not text:
        return "MREL"
    # cot returns JSON, try that first
    try:
        obj = json.loads(text.strip())
        if isinstance(obj, dict) and "label" in obj:
            m = LABEL_RE.search(str(obj["label"]))
            if m:
                return m.group(1).upper()
    except Exception:
        pass
    m = LABEL_RE.search(text)
    if m:
        return m.group(1).upper()
    return "MREL"

def call_sarvam(prompt):
    resp = requests.post(
        SARVAM_URL,
        headers={
            "api-subscription-key": SARVAM_KEY,
            "Content-Type": "application/json",
        },
        json={
            "model": SARVAM_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0,
            "max_tokens": 300,
        },
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"] or ""

def safe_call(fn, prompt, tries=5):
    for i in range(tries):
        try:
            return fn(prompt)
        except Exception as e:
            if i == tries - 1:
                return f"ERROR: {e}"
            wait = 4 * (2 ** i) + random.random()
            print(f"  retry {i+1}: {type(e).__name__}, waiting {wait:.1f}s")
            time.sleep(wait)

DELAY = 2.0

## 5. Runner

In [7]:
OUT = Path("llm_results_sarvam")
OUT.mkdir(exist_ok=True)

def run(df, n=None):
    n = n or len(df)
    path = OUT / f"{SARVAM_MODEL}__cot__n{n}.csv"
    if path.exists():
        cached = pd.read_csv(path)
        if len(cached) >= n:
            print(f"  using cache: {path.name}")
            return cached

    sub = df.head(n)
    rows = []
    for i, r in tqdm(sub.iterrows(), total=len(sub), desc=f"{SARVAM_MODEL}/cot"):
        text = safe_call(call_sarvam, p_cot(r["article"], r["headline"]))
        rows.append({
            "true": r["label"],
            "pred": parse_label(text),
            "raw": (text or "")[:500],
        })
        if (i + 1) % 50 == 0:
            pd.DataFrame(rows).to_csv(path, index=False)
        time.sleep(DELAY)

    out = pd.DataFrame(rows)
    out.to_csv(path, index=False)
    return out

## 6. Smoke test (30 samples)

Verify the model is returning parseable responses before the full run.

In [8]:
smoke = run(test_df, n=30)
print("accuracy:", (smoke["true"] == smoke["pred"]).mean())
print(smoke["pred"].value_counts())
print("\nfirst raw response:")
print(smoke["raw"].iloc[0])

sarvam-105b-conversations/cot:   0%|          | 0/30 [00:00<?, ?it/s]

accuracy: 0.2
pred
HREL    27
MREL     3
Name: count, dtype: int64

first raw response:
{
  "reason": "The article details an incident where a headmistress tied up four primary school students with ropes for misbehaving in class, which aligns directly with the headline questioning if misbehaving is a crime and mentioning the headmistress tying students with ropes.",
  "label": "HREL"
}


## 7. Full run (400 stratified samples)

In [9]:
N = 400
per_class = N // 3

eval_df = (test_df.groupby("label", group_keys=False)
                  .apply(lambda g: g.sample(min(len(g), per_class), random_state=42))
                  .reset_index(drop=True))

print(f"running on {len(eval_df)} samples:")
print(eval_df["label"].value_counts())

results = run(eval_df, n=len(eval_df))
print("\ndone.")

running on 399 samples:
label
HREL    133
LREL    133
MREL    133
Name: count, dtype: int64


/tmp/ipykernel_5928/1599760429.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), per_class), random_state=42))


sarvam-105b-conversations/cot:   0%|          | 0/399 [00:00<?, ?it/s]


done.


## 8. Scores

In [10]:
yt = results["true"].map(lab2id).values
yp = results["pred"].map(lab2id).fillna(1).astype(int).values

f1_each = f1_score(yt, yp, labels=[0, 1, 2], average=None)
f1_w = f1_score(yt, yp, average="weighted")
f1_m = f1_score(yt, yp, average="macro")

print("Sarvam-105b / cot results")
print("-" * 40)
print(f"HREL F1: {f1_each[0]:.3f}")
print(f"MREL F1: {f1_each[1]:.3f}")
print(f"LREL F1: {f1_each[2]:.3f}")
print(f"W F1:    {f1_w:.3f}")
print(f"M F1:    {f1_m:.3f}")

cm = confusion_matrix(yt, yp, labels=[0, 1, 2])
print("\nconfusion matrix:")
print("           pred HREL  pred MREL  pred LREL")
for i, lab in enumerate(LABELS):
    print(f"true {lab}:  {cm[i,0]:>8}   {cm[i,1]:>8}   {cm[i,2]:>8}")

Sarvam-105b / cot results
----------------------------------------
HREL F1: 0.532
MREL F1: 0.138
LREL F1: 0.163
W F1:    0.278
M F1:    0.278

confusion matrix:
           pred HREL  pred MREL  pred LREL
true HREL:       127          5          1
true MREL:       120         12          1
true LREL:        97         24         12


## 9. Comparison with baselines and other prompts

In [11]:
ml_base   = {"HREL": 0.61, "MREL": 0.52, "LREL": 0.60, "W": 0.57, "M": 0.58}
bert_best = {"HREL": 0.68, "MREL": 0.57, "LREL": 0.64, "W": 0.62, "M": 0.63}
sarvam_zs = {"HREL": 0.54, "MREL": 0.14, "LREL": 0.04, "W": 0.24, "M": 0.24}

sarvam_cot = {
    "HREL": round(f1_each[0], 2),
    "MREL": round(f1_each[1], 2),
    "LREL": round(f1_each[2], 2),
    "W":    round(f1_w, 2),
    "M":    round(f1_m, 2),
}

line = "=" * 90
print(line)
print("COMPARISON TABLE")
print(line)
print(f"{'method':<48} {'HREL':>6} {'MREL':>6} {'LREL':>6} {'W':>6} {'M':>6}")
print("-" * 90)

def pr(name, d):
    print(f"{name:<48} {d['HREL']:>6.2f} {d['MREL']:>6.2f} {d['LREL']:>6.2f} {d['W']:>6.2f} {d['M']:>6.2f}")

pr("ML baseline (LR, 5 features)", ml_base)
pr("Paper BERT (mDeBERTa)", bert_best)
print("-" * 90)
pr("Sarvam-105b / zeroshot", sarvam_zs)
pr("Sarvam-105b / cot", sarvam_cot)
print(line)

COMPARISON TABLE
method                                             HREL   MREL   LREL      W      M
------------------------------------------------------------------------------------------
ML baseline (LR, 5 features)                       0.61   0.52   0.60   0.57   0.58
Paper BERT (mDeBERTa)                              0.68   0.57   0.64   0.62   0.63
------------------------------------------------------------------------------------------
Sarvam-105b / zeroshot                             0.54   0.14   0.04   0.24   0.24
Sarvam-105b / cot                                  0.53   0.14   0.16   0.28   0.28


### Ensemble

In [13]:
import pandas as pd
from pathlib import Path
from collections import Counter
from sklearn.metrics import f1_score, confusion_matrix

root = Path("llm_results_sarvam")

zs  = pd.read_csv("sarvam-105b-conversations__zeroshot__n399.csv")
cot = pd.read_csv("sarvam-105b-conversations__cot__n399.csv")
fs  = pd.read_csv("sarvam-105b-conversations__fewshot__n399.csv")

LABELS = ["HREL", "MREL", "LREL"]
lab2id = {"HREL": 0, "MREL": 1, "LREL": 2}

assert len(zs) == len(cot) == len(fs), "CSVs have different lengths"
assert (zs["true"] == cot["true"]).all() and (zs["true"] == fs["true"]).all(), "true labels mismatch"

# Two voting strategies
def rank_vote(z, c, f):
    """Prefer rare classes: LREL > MREL > HREL."""
    labels = {z, c, f}
    if "LREL" in labels:
        return "LREL"
    if "MREL" in labels:
        return "MREL"
    return "HREL"

def majority_vote(z, c, f):
    """Plain majority; tie-break with CoT."""
    counts = Counter([z, c, f])
    top = counts.most_common(1)[0]
    if top[1] >= 2:
        return top[0]
    return c  # tie-break: use CoT

true = zs["true"].values
rank_preds = [rank_vote(z, c, f) for z, c, f in zip(zs["pred"], cot["pred"], fs["pred"])]
maj_preds  = [majority_vote(z, c, f) for z, c, f in zip(zs["pred"], cot["pred"], fs["pred"])]

def score(name, preds):
    yt = pd.Series(true).map(lab2id).values
    yp = pd.Series(preds).map(lab2id).fillna(1).astype(int).values
    pc = f1_score(yt, yp, labels=[0, 1, 2], average=None)
    w = f1_score(yt, yp, average="weighted")
    m = f1_score(yt, yp, average="macro")
    print(f"{name:<30} HREL={pc[0]:.2f}  MREL={pc[1]:.2f}  LREL={pc[2]:.2f}  W={w:.2f}  M={m:.2f}")
    return yt, yp

print("Comparison with individual prompts:")
print("-" * 85)
score("zeroshot (baseline)", zs["pred"].tolist())
score("cot", cot["pred"].tolist())
score("fewshot", fs["pred"].tolist())
print("-" * 85)
_, yp_rank = score("ensemble (rank: prefer rare)", rank_preds)
_, yp_maj  = score("ensemble (majority vote)", maj_preds)

print("\nRank-vote confusion matrix:")
print("           pred HREL  pred MREL  pred LREL")
cm = confusion_matrix(pd.Series(true).map(lab2id), yp_rank, labels=[0,1,2])
for i, lab in enumerate(LABELS):
    print(f"true {lab}:  {cm[i,0]:>8}   {cm[i,1]:>8}   {cm[i,2]:>8}")

print("\nMajority-vote confusion matrix:")
cm = confusion_matrix(pd.Series(true).map(lab2id), yp_maj, labels=[0,1,2])
print("           pred HREL  pred MREL  pred LREL")
for i, lab in enumerate(LABELS):
    print(f"true {lab}:  {cm[i,0]:>8}   {cm[i,1]:>8}   {cm[i,2]:>8}")

Comparison with individual prompts:
-------------------------------------------------------------------------------------
zeroshot (baseline)            HREL=0.54  MREL=0.15  LREL=0.04  W=0.24  M=0.24
cot                            HREL=0.53  MREL=0.14  LREL=0.16  W=0.28  M=0.28
fewshot                        HREL=0.58  MREL=0.25  LREL=0.04  W=0.29  M=0.29
-------------------------------------------------------------------------------------
ensemble (rank: prefer rare)   HREL=0.58  MREL=0.27  LREL=0.19  W=0.35  M=0.35
ensemble (majority vote)       HREL=0.55  MREL=0.16  LREL=0.04  W=0.25  M=0.25

Rank-vote confusion matrix:
           pred HREL  pred MREL  pred LREL
true HREL:       125          7          1
true MREL:       101         30          2
true LREL:        70         49         14

Majority-vote confusion matrix:
           pred HREL  pred MREL  pred LREL
true HREL:       131          2          0
true MREL:       118         15          0
true LREL:        97         33   

### emsemble with zeroshot anti bias

In [ ]:
ab = pd.read_csv("llm_results_sarvam/sarvam-105b-conversations__antibias_zeroshot__n399.csv")

def rank_vote4(z, c, f, a):
    labels = {z, c, f, a}
    if "LREL" in labels: return "LREL"
    if "MREL" in labels: return "MREL"
    return "HREL"

preds = [rank_vote4(z, c, f, a) for z, c, f, a in zip(
    zs["pred"], cot["pred"], fs["pred"], ab["pred"]
)]

# ... then score as before